In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import time, tracemalloc
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [3]:
df_churn = pd.read_csv("/home/huzaifa/Desktop/my_ml_project/data/telecom_churn.csv")   # adjust to your actual filename

In [4]:
df_churn

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [5]:
df_churn.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [6]:
df_churn["TotalCharges"] = pd.to_numeric(df_churn["TotalCharges"], errors="coerce")
df_churn = df_churn.dropna()

In [7]:
if "customerID" in df_churn.columns:
    df_churn = df_churn.drop(columns=["customerID"])

In [8]:
df_churn.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [9]:
df_churn["Churn"] = df_churn["Churn"].map({"Yes": 1, "No": 0})
df_churn

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1


In [10]:
df_churn = pd.get_dummies(df_churn, drop_first=True)

In [11]:
y_churn = df_churn["Churn"].values
X_churn = df_churn.drop(columns=["Churn"]).values
df_churn

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,0,True,True,True,True,False,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,0,False,True,True,True,False,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,0,False,True,True,False,True,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,1,True,True,False,True,False,...,False,False,False,False,False,False,True,False,False,True


In [12]:
X_train_churn, X_test_churn, y_train_churn, y_test_churn = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_churn = scaler.fit_transform(X_train_churn)
X_test_churn = scaler.transform(X_test_churn)

In [13]:
model_svm = LinearSVC(max_iter=5000)

In [14]:
df_churn["Churn"].unique

<bound method Series.unique of 0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7032, dtype: int64>

In [15]:
tracemalloc.start()
start = time.perf_counter()
model_svm.fit(X_train_churn, y_train_churn)
train_time = time.perf_counter() - start
_, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

In [16]:
start = time.perf_counter()
y_pred = model_svm.predict(X_test_churn)
predict_time = time.perf_counter() - start

In [17]:
accuracy = accuracy_score(y_test_churn, y_pred)
f1 = f1_score(y_test_churn, y_pred)
cm = confusion_matrix(y_test_churn, y_pred)

In [18]:
print("Scikit-learn Churn Dataset")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:\n", cm)
print(f"Train time: {train_time:.4f}s")
print(f"Predict time: {predict_time:.4f}s")
print(f"Peak memory: {peak_mem / (1024**2):.2f} MB")

Scikit-learn Churn Dataset
Accuracy: 0.7925
F1 Score: 0.5655
Confusion Matrix:
 [[925 108]
 [184 190]]
Train time: 0.0344s
Predict time: 0.0008s
Peak memory: 0.86 MB


# Tensorflow Implementation

In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models

I0000 00:00:1785165350.433027   17457 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [20]:
y_train_churn_churn_svm = np.where(y_train_churn == 0, -1, 1).astype(np.float32)
y_test_churn_churn_svm = np.where(y_test_churn == 0, -1, 1).astype(np.float32)

In [21]:
model_tf_churn = models.Sequential([
    layers.Input(shape=(X_train_churn.shape[1],)),
    layers.Dense(1)   # no activation — raw score, same as LinearSVC's decision function
])

E0000 00:00:1785165352.211034   17457 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1785165352.211058   17457 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1785165352.211067   17457 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1785165352.211076   17457 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1785165352.211079   17457 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: shahnoor-B550M-AORUS-ELITE
I0000 00:00:1785165352.211082   17457 cuda_diagnostics.cc:183] hostname: shahnoor-B550M-AORUS-ELITE
I0000 00:00:1785165352.211174   17457 cuda_diagnostics.cc:190] libcuda reported version is: 595.71.5
I0000 00:00:1785165352.211192   17457 c

In [22]:
model_tf_churn.compile(optimizer="adam", loss="hinge")

In [23]:
tracemalloc.start()
start = time.perf_counter()
model_tf_churn.fit(X_train_churn, y_train_churn, epochs=50, batch_size=32, verbose=0)
train_time_tf_churn = time.perf_counter() - start
_, peak_mem_tf_churn = tracemalloc.get_traced_memory()
tracemalloc.stop()

In [24]:
start = time.perf_counter()
y_pred_score_tf_churn = model_tf_churn.predict(X_test_churn, verbose=0).flatten()
predict_time_tf_churn = time.perf_counter() - start

In [25]:
y_pred_tf_churn = np.where(y_pred_score_tf_churn > 0, 1, 0)   # convert back to 0/1
y_test_churn_binary = np.where(y_test_churn > 0, 1, 0)
acc_tf_churn = accuracy_score(y_test_churn_binary, y_pred_tf_churn)
f1_tf_churn = f1_score(y_test_churn_binary, y_pred_tf_churn)
print("Tensorflow Churn Dataset")
print(f"Accuracy: {acc_tf_churn:.4f}")
print(f"F1 score: {f1_tf_churn:.4f}")
print(f"Train: {train_time_tf_churn:.4f}s")
print(f"Predict: {predict_time_tf_churn:.4f}s")
print(f"Mem: {peak_mem_tf_churn/(1024**2):.2f}MB")

Tensorflow Churn Dataset
Accuracy: 0.7896
F1 score: 0.5660
Train: 13.7645s
Predict: 0.1311s
Mem: 2.86MB


# Pytorch

In [26]:
"""import torch 
import torch.nn as nn"""

'import torch \nimport torch.nn as nn'

In [27]:
"""X_train_churn_churn_t = torch.tensor(X_train_churn, dtype=torch.float32)
y_train_churn_churn_t = torch.tensor(y_train_churn, dtype=torch.float32).view(-1, 1)
X_test_churn_churn_t = torch.tensor(X_test_churn, dtype=torch.float32)"""

'X_train_churn_churn_t = torch.tensor(X_train_churn, dtype=torch.float32)\ny_train_churn_churn_t = torch.tensor(y_train_churn, dtype=torch.float32).view(-1, 1)\nX_test_churn_churn_t = torch.tensor(X_test_churn, dtype=torch.float32)'

In [28]:
"""model_torch_churn = nn.Linear(X_train_churn.shape[1], 1)
optimizer_churn = torch.optim.Adam(model_torch_churn.parameters(), lr=0.01)"""

'model_torch_churn = nn.Linear(X_train_churn.shape[1], 1)\noptimizer_churn = torch.optim.Adam(model_torch_churn.parameters(), lr=0.01)'

In [29]:
"""def hinge_loss(outputs, labels):
    return torch.mean(torch.clamp(1 - outputs * labels, min=0))"""

'def hinge_loss(outputs, labels):\n    return torch.mean(torch.clamp(1 - outputs * labels, min=0))'

In [30]:
"""tracemalloc.start()
start = time.perf_counter()
for epoch in range(50):
    optimizer_churn.zero_grad()
    outputs = model_torch_churn(X_train_churn_t)
    loss = hinge_loss(outputs, y_train_churn_t)
    loss.backward()
    optimizer_churn.step()
train_time_torch_churn = time.perf_counter() - start
_, peak_mem_torch_churn = tracemalloc.get_traced_memory()
tracemalloc.stop()"""

'tracemalloc.start()\nstart = time.perf_counter()\nfor epoch in range(50):\n    optimizer_churn.zero_grad()\n    outputs = model_torch_churn(X_train_churn_t)\n    loss = hinge_loss(outputs, y_train_churn_t)\n    loss.backward()\n    optimizer_churn.step()\ntrain_time_torch_churn = time.perf_counter() - start\n_, peak_mem_torch_churn = tracemalloc.get_traced_memory()\ntracemalloc.stop()'

In [31]:
"""model_torch_churn.eval()
start = time.perf_counter()
with torch.no_grad():
    y_pred_score_torch_churn = model_torch_churn(X_test_churn_t).numpy().flatten()
predict_time_torch_churn = time.perf_counter() - start"""

'model_torch_churn.eval()\nstart = time.perf_counter()\nwith torch.no_grad():\n    y_pred_score_torch_churn = model_torch_churn(X_test_churn_t).numpy().flatten()\npredict_time_torch_churn = time.perf_counter() - start'

In [32]:
"""y_pred_torch_churn = np.where(y_pred_score_torch_churn > 0, 1, 0)
acc_torch_churn = accuracy_score(y_test_churn_binary, y_pred_torch_churn)
f1_torch_churn = f1_score(y_test_churn_binary, y_pred_torch_churn)
print(f"[Torch-Churn] Acc: {acc_torch_churn:.4f}, F1: {f1_torch_churn:.4f}, Train: {train_time_torch_churn:.4f}s, Predict: {predict_time_torch_churn:.4f}s, Mem: {peak_mem_torch_churn/(1024**2):.2f}MB")"""

'y_pred_torch_churn = np.where(y_pred_score_torch_churn > 0, 1, 0)\nacc_torch_churn = accuracy_score(y_test_churn_binary, y_pred_torch_churn)\nf1_torch_churn = f1_score(y_test_churn_binary, y_pred_torch_churn)\nprint(f"[Torch-Churn] Acc: {acc_torch_churn:.4f}, F1: {f1_torch_churn:.4f}, Train: {train_time_torch_churn:.4f}s, Predict: {predict_time_torch_churn:.4f}s, Mem: {peak_mem_torch_churn/(1024**2):.2f}MB")'